# Multilingual Health Question Answering — Jupyter / Cloud GPU Execution

**Repository:** [SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages](https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages.git)

This notebook automatically clones/downloads the official competition repository, installs required dependencies, and runs fine-tuning and inference for **`facebook/nllb-200-distilled-600M`** with **Hybrid Lexical + Multilingual Dense RAG Retrieval** across 8 African language/country subsets.

In [ ]:
# 1. Download & Extract Repository (Robust Git + Direct Zip Fallback)
import os, sys, shutil, zipfile, urllib.request

repo_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages.git'
zip_url = 'https://github.com/SamAbr/Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages/archive/refs/heads/main.zip'
repo_name = 'Multilingual-Health-Question-Answering-in-Low-Resource-African-Languages'

if not (os.path.exists(repo_name) or os.path.exists(f'{repo_name}-main')):
    # Install git if missing
    !apt-get update -qq && apt-get install -y -qq git || conda install -y -c conda-forge git || true
    
    # Attempt git clone
    clone_status = os.system(f'git clone {repo_url}')
    
    # Fallback to direct Zip download if git is missing or fails
    if clone_status != 0 or not os.path.exists(repo_name):
        print('[INFO] Downloading repository archive directly...')
        urllib.request.urlretrieve(zip_url, 'repo.zip')
        with zipfile.ZipFile('repo.zip', 'r') as zip_ref:
            zip_ref.extractall('.')
        if os.path.exists(f'{repo_name}-main'):
            os.rename(f'{repo_name}-main', repo_name)

if os.path.exists(repo_name):
    %cd {repo_name}
elif os.path.exists(f'{repo_name}-main'):
    %cd {repo_name}-main

!pwd


In [ ]:
# 2. Install Required Dependencies
!pip install -q torch transformers datasets evaluate scikit-learn pandas numpy rouge-score sentence-transformers accelerate peft
print('Dependencies installed successfully!')


In [ ]:
# 3. Check GPU Acceleration
import torch
cuda_avail = torch.cuda.is_available()
print(f'CUDA Available: {cuda_avail}')
if cuda_avail:
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('[WARN] No GPU detected! Fine-tuning will be slow on CPU.')


In [ ]:
# 4. Fast Dry-Run Verification (Testing Pipeline End-to-End)
!python src/nllb_pipeline.py --dry_run --model_name facebook/nllb-200-distilled-600M


In [ ]:
# 5. Run Full NLLB-200 Fine-Tuning & Hybrid RAG Retrieval Pipeline
!python src/nllb_pipeline.py \
    --model_name facebook/nllb-200-distilled-600M \
    --use_dense_rag \
    --epochs 3 \
    --batch_size 8 \
    --learning_rate 5e-5 \
    --submission_path submissions/submission_nllb_hybrid.csv


In [ ]:
# 6. Inspect Generated Submission File
import pandas as pd
sub_path = 'submissions/submission_nllb_hybrid.csv'
sub = pd.read_csv(sub_path)
print(f'Submission shape: {sub.shape}')
display(sub.head(10))


In [ ]:
# 7. Download Submission CSV File
try:
    from google.colab import files
    files.download('submissions/submission_nllb_hybrid.csv')
except ImportError:
    print('Submission saved locally at: submissions/submission_nllb_hybrid.csv')
